In [ ]:
import os 

PATH = "/Volumes/New Volume/malware-detection-dataset/malware-source-code/src"

files = [os.path.join(PATH, file) for file in os.listdir(PATH)]
project_files = [file for file in files if os.path.isdir(file)]

In [ ]:
from collections import Counter 

exts = Counter()
cs_projs = []

for file in project_files:
    files = []
    for root, dirs, contents in os.walk(file, topdown=True):
        files.extend(os.path.splitext(x)[-1] for x in contents)

        if any([x.endswith('.csproj') for x in contents]):
            cs_projs.append(file)

    
    
    exts.update(files)
print(exts)

In [ ]:
import numpy as np 

ext_to_idx_lut = {ext: i for i, ext in enumerate(exts.keys())}
idx_to_ext_lut = {i: ext for ext, i in ext_to_idx_lut.items()}
idx_to_path_lut = {i: path for i, path in enumerate(project_files) }
path_to_idx_lut = {path: i for i, path in idx_to_path_lut.items()}

exts_matrix = np.zeros((len(project_files), len(ext_to_idx_lut)))

for i, file in enumerate(project_files):
    tmp_counter = Counter()
    for root, dirs, contents in os.walk(file):
        tmp_counter.update([os.path.splitext(f)[-1] for f in contents]) 
    
    for ext, freq in tmp_counter.items():
        exts_matrix[i, ext_to_idx_lut[ext]] = freq

    
exts_matrix.shape

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import HDBSCAN

scaled_matrix = StandardScaler().fit_transform(exts_matrix)
embedding = umap.UMAP(
    n_neighbors=40,
    min_dist=0.001,   
    n_components=2,
    random_state=42
).fit_transform(scaled_matrix)

labels = HDBSCAN(
    min_samples=10, 
    min_cluster_size=100,
).fit_predict(embedding)

clustered = (labels >= 0)
plt.scatter(embedding[~clustered, 0], embedding[~clustered, 1], c='grey', alpha=0.5)
plt.scatter(embedding[clustered, 0], embedding[clustered, 1], c=labels[clustered])
plt.show()

In [ ]:
groups = {}
for label in set(labels):
    groups[int(label)] = np.array(project_files)[labels == label].tolist()

In [ ]:
def group_info(id, groups): 
    assert id in groups.keys(), "Group not found in groups dict"

    idxs = [path_to_idx_lut[path] for path in groups[id]]

    files = exts_matrix[idxs]

    stats = {}

    for i in range(exts_matrix.shape[-1]):
        if (files[:, i] == 0).all():
            continue
        else: 
            stats[idx_to_ext_lut[i]] = {
                'mean': np.mean(files[:, i]),
                'std': np.std(files[:, i]),
            }
    return dict(sorted(stats.items(), key=lambda item: item[1]['mean'], reverse=True))
    
   
print(group_info(0, groups))

In [ ]:
groups[2]

In [ ]:
import bs4

for proj in cs_projs:
    files = os.listdir(proj)
    cs_proj_file = None

    for file in files: 
        if file.endswith('.csproj'):
            cs_proj_file = file

    if cs_proj_file is None: 
        continue

    s = bs4.BeautifulSoup(open(os.path.join(proj, cs_proj_file), "r").read(), features='xml')
    print(s)